# MCPify your AWS Lambda with Gateway OAuth Inbound
## Transform AWS Lambda functions into secure MCP tools with Bedrock AgentCore Gateway

## Overview
Bedrock AgentCore Gateway provides customers a way to turn their existing AWS Lambda functions into fully-managed MCP servers without needing to manage infra or hosting. Gateway will provide a uniform Model Context Protocol (MCP) interface across all these tools. Gateway employs a dual authentication model to ensure secure access control for both incoming requests and outbound connections to target resources. The framework consists of two key components: Inbound Auth, which validates and authorizes users attempting to access gateway targets, and Outbound Auth, which enables the gateway to securely connect to backend resources on behalf of authenticated users. Gateways uses IAM role to authorize the calls to AWS Lambda functions for outbound authorization.

In this example, we will demonstrate OAuth for inbound authorization and IAM roles for outbound authorization.

![How does it work](images/lambda-iam-gateway.png)

### Tutorial Details


| Information          | Details                                                   |
|:---------------------|:----------------------------------------------------------|
| Tutorial type        | Interactive                                               |
| AgentCore components | AgentCore Gateway, AgentCore Identity                     |
| Agentic Framework    | Strands Agents                                            |
| Gateway Target type  | AWS Lambda                                                |
| Inbound Auth IdP     | Amazon Cognito                                            |
| Outbound Auth        | AWS IAM                                                   |
| LLM model            | Anthropic Claude Sonnet 3.7, Amazon Nova Pro              |
| Tutorial components  | Creating AgentCore Gateway and Invoking AgentCore Gateway |
| Tutorial vertical    | Cross-vertical                                            |
| Example complexity   | Easy                                                      |
| SDK used             | boto3                                                     |

In the first part of the tutorial we will create some AmazonCore Gateway targets

### Tutorial Architecture
In this tutorial we will transform operations defined in AWS lambda function into MCP tools and host it in Bedrock AgentCore Gateway.
For demonstration purposes, we will use a Strands Agent using Amazon Bedrock models
In our example we will use a very simple agent with two tools: get_order and update_order.

## Prerequisites

To execute this tutorial you will need:
* Jupyter notebook (Python kernel)
* uv
* AWS credentials
* Amazon Cognito

## Configuring Authentication for Incoming AgentCore Gateway Requests
AgentCore Gateway provides secure connections via inbound and outbound authentication. For the inbound authentication, the AgentCore Gateway analyzes the OAuth token passed during invocation to decide allow or deny the access to a tool in the gateway. If a tool needs access to external resources, the AgentCore Gateway can use outbound authentication via API Key, IAM or OAuth Token to allow or deny the access to the external resource.



During the inbound authorization flow, an agent or the MCP client calls an MCP tool in the AgentCore Gateway adding an OAuth access token (generated from the user’s IdP). AgentCore Gateway then validates the OAuth access token and performs inbound authorization.

If the tool running in AgentCore Gateway needs to access external resources, OAuth will retrieve credentials of downstream resources using the resource credential provider for the Gateway target. AgentCore Gateway pass the authorization credentials to the caller to get access to the downstream API. 

In [1]:
!pip install --force-reinstall -U -r requirements.txt --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-instrumentation-aiopg 0.54b1 requires opentelemetry-instrumentation==0.54b1, but you have opentelemetry-instrumentation 0.59b0 which is incompatible.
opentelemetry-instrumentation-urllib3 0.54b1 requires opentelemetry-instrumentation==0.54b1, but you have opentelemetry-instrumentation 0.59b0 which is incompatible.
opentelemetry-instrumentation-urllib3 0.54b1 requires opentelemetry-semantic-conventions==0.54b1, but you have opentelemetry-semantic-conventions 0.59b0 which is incompatible.
opentelemetry-instrumentation-dbapi 0.54b1 requires opentelemetry-instrumentation==0.54b1, but you have opentelemetry-instrumentation 0.59b0 which is incompatible.
opentelemetry-instrumentation-dbapi 0.54b1 requires opentelemetry-semantic-conventions==0.54b1, but you have opentelemetry-semantic-conventions 0.59b0 whic

In [2]:
# Set AWS credentials if not using Amazon SageMaker notebook
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1') # set the AWS region

In [3]:
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '../..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

In [5]:
#### Create a sample AWS Lambda function that you want to convert into MCP tools
lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")

if lambda_resp is not None:
    if lambda_resp['exit_code'] == 0:
        print("Lambda function created with ARN: ", lambda_resp['lambda_function_arn'])
    else:
        print("Lambda function creation failed with message: ", lambda_resp['lambda_function_arn'])

Reading code from zip file
Creating IAM role for lambda function
IAM role gateway_lambda_iamrole already exists. Using the same ARN arn:aws:iam::757120839849:role/gateway_lambda_iamrole
Creating lambda function
AWS Lambda function gateway_lambda already exists. Using the same ARN arn:aws:lambda:us-east-1:757120839849:function:gateway_lambda
Lambda function creation failed with message:  arn:aws:lambda:us-east-1:757120839849:function:gateway_lambda


In [6]:
#### Create an IAM role for the Gateway to assume
import utils
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

Role already exists -- deleting and creating it again
policies: {'PolicyNames': ['AgentCorePolicy'], 'IsTruncated': False, 'ResponseMetadata': {'RequestId': 'a2aaf3fc-6c03-4099-9274-c1ab3daf6ef5', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 17 Nov 2025 23:26:21 GMT', 'x-amzn-requestid': 'a2aaf3fc-6c03-4099-9274-c1ab3daf6ef5', 'content-type': 'text/xml', 'content-length': '380'}, 'RetryAttempts': 0}}
deleting agentcore-sample-lambdagateway-role
recreating agentcore-sample-lambdagateway-role
attaching role policy agentcore-sample-lambdagateway-role
Agentcore gateway role ARN:  arn:aws:iam::757120839849:role/agentcore-sample-lambdagateway-role


# Create Amazon Cognito Pool for Inbound authorization to Gateway

In [7]:
# Creating Cognito User Pool 
import os
import boto3
import requests
import time
from botocore.exceptions import ClientError

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"}
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret  = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

# Get discovery URL  
cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration'
print(cognito_discovery_url)

Creating or retrieving Cognito resources...
Found domain for user pool us-east-1_mVE1eCmoL: us-east-1mve1ecmol (https://us-east-1mve1ecmol.auth.us-east-1.amazoncognito.com)
User Pool ID: us-east-1_mVE1eCmoL
Resource server ensured.
Client ID: 594pc7cfc0jra2kapmps2em61f
https://cognito-idp.us-east-1.amazonaws.com/us-east-1_mVE1eCmoL/.well-known/openid-configuration


# Create the Gateway with Amazon Cognito Authorizer for inbound authorization

In [8]:
# CreateGateway with Cognito authorizer without CMK. Use the Cognito user pool created in the previous step
gateway_client = boto3.client('bedrock-agentcore-control', region_name = os.environ['AWS_DEFAULT_REGION'])
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [client_id],  # Client MUST match with the ClientId configured in Cognito. Example: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(name='TestGWforLambda',
    roleArn = agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway 
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config, 
    description='AgentCore Gateway with AWS Lambda target type'
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)
time.sleep(10)

{'ResponseMetadata': {'RequestId': 'fff10c64-587e-4b13-afea-3daf1f5d35b4', 'HTTPStatusCode': 202, 'HTTPHeaders': {'date': 'Mon, 17 Nov 2025 23:26:31 GMT', 'content-type': 'application/json', 'content-length': '815', 'connection': 'keep-alive', 'x-amzn-requestid': 'fff10c64-587e-4b13-afea-3daf1f5d35b4', 'x-amzn-remapped-x-amzn-requestid': '28d95a26-da80-4bde-bc8b-03e8f5a8928c', 'x-amzn-remapped-content-length': '815', 'x-amzn-remapped-connection': 'keep-alive', 'x-amz-apigw-id': 'UNhOLGHNIAMEptA=', 'x-amzn-trace-id': 'Root=1-691baf27-5cbbb8946ee4713471292cef', 'x-amzn-remapped-date': 'Mon, 17 Nov 2025 23:26:31 GMT'}, 'RetryAttempts': 0}, 'gatewayArn': 'arn:aws:bedrock-agentcore:us-east-1:757120839849:gateway/testgwforlambda-pcb4kgki0t', 'gatewayId': 'testgwforlambda-pcb4kgki0t', 'gatewayUrl': 'https://testgwforlambda-pcb4kgki0t.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp', 'createdAt': datetime.datetime(2025, 11, 17, 23, 26, 31, 304180, tzinfo=tzutc()), 'updatedAt': datetime.d

# Create an AWS Lambda target and transform into MCP tools

In [9]:
# Replace the AWS Lambda function ARN below
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_resp['lambda_function_arn'], # Replace this with your AWS Lambda function ARN
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "tool to get the order",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    },                    
                    {
                        "name": "update_order_tool",
                        "description": "tool to update the orderId",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    }
                ]
            }
        }
    }
}

credential_config = [ 
    {
        "credentialProviderType" : "GATEWAY_IAM_ROLE"
    }
]
targetname='LambdaUsingSDK'
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='Lambda Target using SDK',
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config)

# Calling Bedrock AgentCore Gateway from a Strands Agent

The Strands agent seamlessly integrates with AWS tools through the Bedrock AgentCore Gateway, which implements the Model Context Protocol (MCP) specification. This integration enables secure, standardized communication between AI agents and AWS services.

At its core, the Bedrock AgentCore Gateway serves as a protocol-compliant Gateway that exposes fundamental MCP APIs: ListTools and InvokeTools. These APIs allow any MCP-compliant client or SDK to discover and interact with available tools in a secure, standardized way. When the Strands agent needs to access AWS services, it communicates with the Gateway using these MCP-standardized endpoints.

The Gateway's implementation adheres strictly to the (MCP Authorization specification)[https://modelcontextprotocol.org/specification/draft/basic/authorization], ensuring robust security and access control. This means that every tool invocation by the Strands agent goes through authorization step, maintaining security while enabling powerful functionality.

For example, when the Strands agent needs to access MCP tools, it first calls ListTools to discover available tools, then uses InvokeTools to execute specific actions. The Gateway handles all the necessary security validations, protocol translations, and service interactions, making the entire process seamless and secure.

This architectural approach means that any client or SDK that implements the MCP specification can interact with AWS services through the Gateway, making it a versatile and future-proof solution for AI agent integrations.

![Strands agent calling Gateway](images/strands-lambda-gateway.png)

# Request the access token from Amazon Cognito for inbound authorization

In [10]:
import time
time.sleep(10)

In [11]:
print("Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes")
token_response = utils.get_token(user_pool_id, client_id, client_secret,scopeString,REGION)
token = token_response["access_token"]
print("Token response:", token)

Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes
594pc7cfc0jra2kapmps2em61f
Token response: eyJraWQiOiJNUnNEWmhabU1MYVhUYTl5bFFmZTQ4b2lYUUpzSjBVSjZyRWJ1aDVcL0R4ND0iLCJhbGciOiJSUzI1NiJ9.eyJzdWIiOiI1OTRwYzdjZmMwanJhMmthcG1wczJlbTYxZiIsInRva2VuX3VzZSI6ImFjY2VzcyIsInNjb3BlIjoic2FtcGxlLWFnZW50Y29yZS1nYXRld2F5LWlkXC9nYXRld2F5OndyaXRlIHNhbXBsZS1hZ2VudGNvcmUtZ2F0ZXdheS1pZFwvZ2F0ZXdheTpyZWFkIiwiYXV0aF90aW1lIjoxNzYzNDIyMDEyLCJpc3MiOiJodHRwczpcL1wvY29nbml0by1pZHAudXMtZWFzdC0xLmFtYXpvbmF3cy5jb21cL3VzLWVhc3QtMV9tVkUxZUNtb0wiLCJleHAiOjE3NjM0MjU2MTIsImlhdCI6MTc2MzQyMjAxMiwidmVyc2lvbiI6MiwianRpIjoiZTgyNDdlMjYtNjgyYi00MTlkLTgxZGUtZTJlOGYwNTBmYmI4IiwiY2xpZW50X2lkIjoiNTk0cGM3Y2ZjMGpyYTJrYXBtcHMyZW02MWYifQ.LjOjemEPaqGxblJK0qJsK7BvPK1cZv_kXgUkkOBq5U-xFRBW5AtBWkn_T7FEvZdqA6Gd5B9xK0P25xL-w62aIdlJNGgrQW53OXmzLLapvM1qI0eu2uAgruagFOrns4TfirtqqLwMY641_F95b7Els_E-Sg-fT1ttWgH0d_EGzSYtqlb9npSUm_ojwWdS_ydJ0KAKOCo1JP42BU_BNjeYoVkI818Pl2qKTC

# LiteLLM calling MCP tools of AWS Lambda using Bedrock AgentCore Gateway

In [ ]:
import asyncio
import os
from litellm.experimental_mcp_client.client import MCPClient
from mcp.types import CallToolRequestParams

In [ ]:
# Initialize LiteLLM MCP client with HTTP transport and Bearer token auth
client = MCPClient(
    server_url=gatewayURL,
    transport_type="http",
    auth_type="bearer_token",
    auth_value=token,
)

print("MCP Client initialized successfully!")

INFO | strands.telemetry.metrics | Creating Strands MetricsClient


Tools loaded in the agent are ['LambdaUsingSDK___get_order_tool', 'LambdaUsingSDK___update_order_tool']
<thinking> The user has requested a list of all available tools. I should provide this information directly as it is part of the system's capabilities. </thinking>

Here are the tools available to me:

1. **LambdaUsingSDK___get_order_tool**
   - Description: Tool to get the order.
   - Parameters:
     - `orderId` (string): Property orderId.
   - Required: `orderId`.

2. **LambdaUsingSDK___update_order_tool**
   - Description: Tool to update the orderId.
   - Parameters:
     - `orderId` (string): Property orderId.
   - Required: `orderId`.<thinking> The user has requested to check the order status for a specific order ID. I should use the `LambdaUsingSDK___get_order_tool` to retrieve the order status. </thinking>


Tool #1: LambdaUsingSDK___get_order_tool
<thinking> The tool has returned the order status for order ID 123. I should present this information to the user. </thinking>

T

In [ ]:
print("Connecting to AgentCore Gateway MCP server...")
await client.connect()
print("\n✓ Connected successfully!")
print("\nAvailable Tools:")
print("-" * 50)

# List all available tools
tools = await client.list_tools()

if tools:
    for i, tool in enumerate(tools, 1):
        print(f"\n{i}. {tool.name}")
        print(f"   Description: {tool.description}")
        if hasattr(tool, 'inputSchema'):
            print(f"   Input Schema: {tool.inputSchema}")
else:
    print("No tools available.")

print("\n" + "-" * 50)
print(f"Total tools: {len(tools)}")


# Call the __get_order_tool with orderId argument
tool_name = targetname + "___get_order_tool"

print("\n" + "=" * 50)
print(f"Calling {tool_name} with orderId=123...")
print("=" * 50)

# Create the CallToolRequestParams object
call_params = CallToolRequestParams(
    name=tool_name,
    arguments={"orderId": "123"}
)

tool_result = await client.call_tool(call_params)

print(f"\nTool Result:")
print(f"{tool_result}")

**Issue: if you get below error while executing below cell, it indicates incompatibily between pydantic and pydantic-core versions.**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**How to resolve?**

You will need to make sure you have pydantic==2.7.2 and pydantic-core 2.27.2 that are both compatible. Restart the kernel once done.

# Clean up

Additional resources are also created like IAM role, IAM Policies, Credentials provider, AWS Lambda functions, Cognito user pools, s3 buckets that you might need to manually delete as part of the clean up. This depends on the example you run.

## Delete the gateway (Optional)

In [ ]:
import utils
#utils.delete_gateway(gateway_client,gatewayID)